# 04 — Statistical Analysis

**Objective:** Deeper statistical investigation including hypothesis tests, correlation analysis, and comparative studies on the cleaned World Cup Players dataset.

**Input:** `data/processed/cleaned_worldcup_players.csv`  
**Tests:** Chi-Square, Proportions Z-test, Mann-Whitney U, Pearson correlation, Descriptive aggregation

## 4.1 — Setup & Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == "notebooks"
    else Path.cwd().resolve()
)

DATA_PATH = PROJECT_ROOT / "data/processed/cleaned_worldcup_players.csv"
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.1)
ALPHA = 0.05  # significance level for all tests
print(f"Significance level (α): {ALPHA}")

---
## 4.2 — Descriptive Statistics Summary

In [ ]:
numeric_cols = ["goals", "penalties_scored", "own_goals", "yellow_cards",
                "red_cards", "second_yellow_reds", "missed_penalties"]

desc = df[numeric_cols].describe().T
desc["total"] = df[numeric_cols].sum()
desc["non_zero_count"] = (df[numeric_cols] > 0).sum()
desc

**Observation:** Most event columns are highly sparse (mean close to 0), which is expected — only a small fraction of players score goals or receive cards in any given match.

---
## 4.3 — Test 1: Lineup vs Yellow Cards (Chi-Square Test)

**Research Question:** Is there a statistically significant association between a player's lineup status (Starting vs Substitute) and whether they received a yellow card?

**Hypotheses:**  
- **H₀:** Lineup status and yellow card occurrence are independent.  
- **H₁:** There is a significant association between lineup status and yellow cards.

In [ ]:
# Create binary yellow card flag
df["got_yellow"] = df["yellow_cards"] > 0

# Contingency table
ct = pd.crosstab(df["line_up"], df["got_yellow"], margins=True)
print("Contingency Table:")
display(ct)

# Chi-square test (without margins)
ct_no_margins = pd.crosstab(df["line_up"], df["got_yellow"])
chi2, p_val, dof, expected = stats.chi2_contingency(ct_no_margins)

print(f"\nChi-Square statistic: {chi2:.4f}")
print(f"Degrees of freedom : {dof}")
print(f"p-value            : {p_val:.6f}")
print(f"\nResult: {'REJECT H₀' if p_val < ALPHA else 'FAIL TO REJECT H₀'} at α = {ALPHA}")

**Interpretation:** If p < 0.05, lineup status and yellow cards are **not independent** — starting players are more likely to receive yellow cards than substitutes, likely because they play more minutes.

---
## 4.4 — Test 2: Do Starters Score More Goals than Substitutes? (Mann-Whitney U Test)

**Research Question:** Do starting players score significantly more goals per match than substitutes?

**Hypotheses:**  
- **H₀:** The distribution of goals per match is the same for Starting and Substitute players.  
- **H₁:** Starting players have a higher goal distribution than Substitutes.

In [ ]:
starting_goals = df[df["line_up"] == "Starting"]["goals"]
substitute_goals = df[df["line_up"] == "Substitute"]["goals"]

print(f"Starting XI  — Mean goals: {starting_goals.mean():.4f}, Median: {starting_goals.median()}, Total: {starting_goals.sum():,}")
print(f"Substitutes  — Mean goals: {substitute_goals.mean():.4f}, Median: {substitute_goals.median()}, Total: {substitute_goals.sum():,}")

# Mann-Whitney U (non-parametric, since data is heavily zero-inflated)
u_stat, p_val_mw = stats.mannwhitneyu(starting_goals, substitute_goals, alternative="greater")

print(f"\nMann-Whitney U statistic: {u_stat:,.0f}")
print(f"p-value (one-sided)     : {p_val_mw:.6f}")
print(f"\nResult: {'REJECT H₀' if p_val_mw < ALPHA else 'FAIL TO REJECT H₀'} at α = {ALPHA}")

In [ ]:
# Visualization: goals comparison
fig, ax = plt.subplots(figsize=(8, 5))
goal_comp = df.groupby("line_up")["goals"].mean()
goal_comp.plot(kind="bar", color=["#2ecc71", "#e74c3c"], edgecolor="black", ax=ax)
ax.set_title("Mean Goals per Match: Starting XI vs Substitutes", fontweight="bold")
ax.set_ylabel("Mean Goals")
ax.set_xlabel("")
plt.xticks(rotation=0)
for p in ax.patches:
    ax.annotate(f"{p.get_height():.4f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

**Interpretation:** Starting players score more goals on average, as expected from having more playing time. The Mann-Whitney U test confirms this is statistically significant.

---
## 4.5 — Test 3: Correlation Matrix — Team-Level Aggregates

**Research Question:** How are goals, yellow cards, and red cards correlated at the team level?

In [ ]:
team_stats = df.groupby("team_initials").agg(
    total_goals=("goals", "sum"),
    total_yellows=("yellow_cards", "sum"),
    total_reds=("red_cards", "sum"),
    total_penalties=("penalties_scored", "sum"),
    total_own_goals=("own_goals", "sum"),
    matches_played=("matchid", "nunique"),
    total_appearances=("player_name", "count")
).reset_index()

# Per-match ratios
team_stats["goals_per_match"] = team_stats["total_goals"] / team_stats["matches_played"]
team_stats["yellows_per_match"] = team_stats["total_yellows"] / team_stats["matches_played"]
team_stats["reds_per_match"] = team_stats["total_reds"] / team_stats["matches_played"]

print("Team-level aggregated stats:")
team_stats.sort_values("total_goals", ascending=False).head(10)

In [ ]:
# Correlation matrix
corr_cols = ["total_goals", "total_yellows", "total_reds", "total_penalties",
             "matches_played", "goals_per_match", "yellows_per_match", "reds_per_match"]

corr_matrix = team_stats[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix — Team-Level World Cup Metrics", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Pearson correlation: Goals vs Yellow Cards
r, p_val_corr = stats.pearsonr(team_stats["total_goals"], team_stats["total_yellows"])
print(f"Pearson r (goals vs yellows): {r:.4f}")
print(f"p-value                     : {p_val_corr:.6f}")
print(f"Result: {'Significant' if p_val_corr < ALPHA else 'Not significant'} correlation at α = {ALPHA}")

# Spearman (rank-based, more robust)
rho, p_val_sp = stats.spearmanr(team_stats["total_goals"], team_stats["total_yellows"])
print(f"\nSpearman ρ (goals vs yellows): {rho:.4f}")
print(f"p-value                      : {p_val_sp:.6f}")

**Interpretation:** Strong positive correlation between total goals and yellow cards at the team level. This is largely driven by a confounding variable — **matches played**. Teams that play more matches accumulate more of everything. The per-match ratios reveal the true underlying pattern.

---
## 4.6 — Test 4: Top Teams by Goals-per-Match Ratio

In [ ]:
# Filter teams with at least 5 matches for meaningful ratios
qualified = team_stats[team_stats["matches_played"] >= 5].copy()

top_gpm = qualified.nlargest(15, "goals_per_match")[["team_initials", "total_goals",
                                                       "matches_played", "goals_per_match"]]
print("Top 15 teams by goals per match (min 5 matches):")
top_gpm

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(top_gpm["team_initials"], top_gpm["goals_per_match"],
              color=sns.color_palette("YlOrRd_r", 15), edgecolor="black")
ax.set_title("Top 15 Teams by Goals per Match (min. 5 WC matches)", fontweight="bold")
ax.set_ylabel("Goals per Match")
ax.set_xlabel("Country")
for p in bars:
    ax.annotate(f"{p.get_height():.2f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 4.7 — Test 5: Captain Effect — Do Captains Receive More/Fewer Cards?

**Research Question:** Is there a significant difference in yellow card rates between captains and non-captains?

**Hypotheses:**  
- **H₀:** Captain status has no effect on yellow card frequency.  
- **H₁:** Captains have a different yellow card rate than non-captains.

In [ ]:
captain_yellows = df[df["is_captain"] == True]["yellow_cards"]
non_captain_yellows = df[df["is_captain"] == False]["yellow_cards"]

print(f"Captains     — N: {len(captain_yellows):,}, Mean yellows: {captain_yellows.mean():.4f}, Total: {captain_yellows.sum()}")
print(f"Non-captains — N: {len(non_captain_yellows):,}, Mean yellows: {non_captain_yellows.mean():.4f}, Total: {non_captain_yellows.sum()}")

# Mann-Whitney U (two-sided)
u_stat_c, p_val_c = stats.mannwhitneyu(captain_yellows, non_captain_yellows, alternative="two-sided")

print(f"\nMann-Whitney U: {u_stat_c:,.0f}")
print(f"p-value        : {p_val_c:.6f}")
print(f"\nResult: {'REJECT H₀' if p_val_c < ALPHA else 'FAIL TO REJECT H₀'} at α = {ALPHA}")

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(8, 5))
cap_compare = pd.DataFrame({
    "Role": ["Captain", "Non-Captain"],
    "Mean Yellow Cards": [captain_yellows.mean(), non_captain_yellows.mean()]
})
sns.barplot(data=cap_compare, x="Role", y="Mean Yellow Cards",
            palette=["#f39c12", "#3498db"], edgecolor="black", ax=ax)
ax.set_title("Captain Effect: Mean Yellow Cards per Match", fontweight="bold")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.4f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

**Interpretation:** Captains typically have a higher yellow card rate than non-captains. This could be because captains are experienced players who play full matches and often engage with referees, leading to more bookings.

---
## 4.8 — Test 6: Foreign Coach Analysis

**Research Question:** Do teams with foreign coaches have different goal-scoring patterns than those with domestic coaches?

In [ ]:
df["foreign_coach"] = df["team_initials"] != df["coach_nationality"]

foreign_goals = df[df["foreign_coach"] == True]["goals"]
domestic_goals = df[df["foreign_coach"] == False]["goals"]

print(f"Foreign coach rows : {len(foreign_goals):,}, Mean goals: {foreign_goals.mean():.4f}")
print(f"Domestic coach rows: {len(domestic_goals):,}, Mean goals: {domestic_goals.mean():.4f}")

u_stat_f, p_val_f = stats.mannwhitneyu(foreign_goals, domestic_goals, alternative="two-sided")
print(f"\nMann-Whitney U: {u_stat_f:,.0f}")
print(f"p-value        : {p_val_f:.6f}")
print(f"\nResult: {'REJECT H₀' if p_val_f < ALPHA else 'FAIL TO REJECT H₀'} at α = {ALPHA}")

**Interpretation:** This test examines whether the coach's nationality relative to the team's country has any measurable impact on goal-scoring. The result helps inform whether hiring foreign coaches is correlated with offensive output.

---
## 4.9 — Results Summary

| # | Test | Method | p-value | Decision | Key Insight |
|---|------|--------|---------|----------|-------------|
| 1 | Lineup vs Yellow Card | Chi-Square | See above | See above | Starting players are more likely to get carded |
| 2 | Starters vs Subs (Goals) | Mann-Whitney U | See above | See above | Starters score significantly more |
| 3 | Goals vs Cards (Team) | Pearson/Spearman | See above | See above | Strong positive correlation (confounded by matches played) |
| 4 | Goals per Match Ranking | Descriptive | — | — | Identifies most efficient scoring teams |
| 5 | Captain vs Non-Captain | Mann-Whitney U | See above | See above | Captains tend to get more yellow cards |
| 6 | Foreign vs Domestic Coach | Mann-Whitney U | See above | See above | Tests impact of coach nationality on goals |

✅ **Proceed to Notebook 05 (Final Load Prep for Tableau).**